In [23]:
import json
import sys
from pathlib import Path

import torch
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

In [24]:
# Resolve paths from either the repository root or the model/ folder.
repo_root = Path.cwd()
if not (repo_root / "keys" / "keys.json").exists() and repo_root.name == "model":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from model.bt_validator import BTValidationError, BehaviorTreeValidator

keys_path = repo_root / "keys" / "keys.json"
with keys_path.open("r", encoding="utf-8") as f:
    keys = json.load(f)

login(token=keys["HF_TOKEN"])
model_id = keys["HF_LLAMA_FT_MODEL"]
output_tree_path = repo_root / "model" / "tree.xml"
vocabulary_path = repo_root / "model" / "allowed_primitives.yaml"
validator = BehaviorTreeValidator.from_yaml(vocabulary_path)

In [25]:
# Use bf16 when the active CUDA device supports it; otherwise fall back to fp16.
if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
    compute_dtype = torch.bfloat16
else:
    compute_dtype = torch.float16

# 4-bit quantization keeps inference lightweight for the released 1B model.
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
)

In [26]:
# The HF token from keys/keys.json is used above for authentication.
# No interactive Hugging Face login is required when HF_TOKEN is set.

In [27]:
# Load the fine-tuned BTGenBot-2 model from the Hugging Face repo in HF_LLAMA_FT_MODEL.
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=quantization_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/147 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


In [28]:
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
)

# Keep generation bounded to one behavior tree while allowing the model to sample valid alternatives.
generation_args = {
    "max_new_tokens": 500,
    "return_full_text": False,
    "do_sample": True,
}

In [29]:
system_content = """
You are a helpful assistant that can assist with creating behavior trees.
Your task:
- Convert the provided summary of a behavior into an XML-formatted behavior tree.
- Ensure the behavior tree matches the description in the summary.
- The behavior tree must be compatible with the BehaviorTree.CPP library.
- Only use the actions and parameters provided in the action list below the summary.

Output Requirements:
- Output only the XML representation of the behavior tree. Do not include explanations, comments, or any additional text.
- Ensure all actions and parameters strictly match the provided list.
- If possible limit the use of SubTrees.

Please generate the behavior tree based on the summary and action list provided.
"""

In [30]:
user_content = """
This behavior tree manages a robot doing navigation. Initially, it moves to the location "Warehouse Left". Then, it moves to the "Warehouse Forklift".

Actions: [MoveTo (parameters: location)]
"""

messages = [
    {"role": "system", "content": system_content},
    {"role": "user", "content": user_content},
]

In [31]:
print(f"Repository root: {repo_root}")
print(f"Allowed primitives: {vocabulary_path}")
print(f"Generated tree will be saved to: {output_tree_path}")

Repository root: /home/riccardo/BTGenBot-2
Allowed primitives: /home/riccardo/BTGenBot-2/model/allowed_primitives.yaml
Generated tree will be saved to: /home/riccardo/BTGenBot-2/model/tree.xml


In [32]:
# Reject invalid generations and sample again, as described by the inference-time validator.
max_validation_attempts = 3
validation_errors = []

for attempt in range(1, max_validation_attempts + 1):
    output = pipe(messages, **generation_args)
    generated_text = output[0]["generated_text"]
    print(f"Generation attempt {attempt}:\n{generated_text}")

    try:
        validator.validate(generated_text)
    except BTValidationError as error:
        validation_errors.append(f"Attempt {attempt}: {error}")
        print(f"Rejected: {error}")
        continue

    final_tree = generated_text.strip()
    print(f"Accepted behavior tree on attempt {attempt}.")
    break
else:
    details = "\n".join(validation_errors)
    raise BTValidationError(
        f"No valid behavior tree generated after {max_validation_attempts} attempts:\n{details}"
    )

print(final_tree)

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generation attempt 1:
<root BTCPP_format="4" main_tree_to_execute="MainTree">
    <BehaviorTree ID="MainTree">
        <Sequence>
            <MoveTo location="Warehouse Left"/>
            <MoveTo location="Warehouse Forklift"/>
        </Sequence>
    </BehaviorTree>
</root>
Accepted behavior tree on attempt 1.
<root BTCPP_format="4" main_tree_to_execute="MainTree">
    <BehaviorTree ID="MainTree">
        <Sequence>
            <MoveTo location="Warehouse Left"/>
            <MoveTo location="Warehouse Forklift"/>
        </Sequence>
    </BehaviorTree>
</root>


In [33]:
# Save the generated behavior tree for simulator or BehaviorTree.CPP use.
output_tree_path.parent.mkdir(parents=True, exist_ok=True)
with output_tree_path.open("w", encoding="utf-8") as f:
    f.write(final_tree)